# Fresh import-feedback portability replay

Private CPU-only/offline execution of the unchanged eight-condition synthetic experiment. No competition snapshots, evaluator or model.

In [ ]:
from pathlib import Path
import hashlib,json,subprocess,sys,platform
ROOT=Path('/kaggle/working/fresh_feedback_portability')
ROOT.mkdir(exist_ok=False)
PAYLOAD={'source_import_guard.py': '"""Budgeted, offline source provenance check executed by an agent skill.\n\nThis standalone file uses ordinary subprocesses inside the supplied sandbox.\nIt never imports or changes an evaluator, a reference patch, or test assertions.\n"""\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\n\ndef git(root, *args):\n    return subprocess.run([\'git\', \'-C\', str(root), *args], capture_output=True,\n                          timeout=10, check=False)\n\n\ndef discover_workspace(explicit=None):\n    if explicit and (str(explicit) != \'/workspace\' or Path(explicit).is_dir()):\n        candidates = [Path(explicit)]\n    else:\n        candidates = [Path.cwd(), Path(\'/workspace\'),\n                      Path(tempfile.gettempdir()).parent / \'workspace\']\n        candidates.extend(Path(p) for p in os.environ.get(\'PYTHONPATH\', \'\').split(os.pathsep) if p)\n    roots = set()\n    for candidate in candidates:\n        if not candidate.is_dir():\n            continue\n        result = git(candidate, \'rev-parse\', \'--show-toplevel\')\n        if result.returncode == 0:\n            roots.add(Path(result.stdout.decode().strip()).resolve(strict=True))\n    if len(roots) != 1:\n        raise ValueError(\'Cannot identify one Git workspace; pass --workspace\')\n    return roots.pop()\n\n\ndef source_location(root, module):\n    if not re.fullmatch(r\'[A-Za-z_]\\w*(?:\\.[A-Za-z_]\\w*)*\', module):\n        raise ValueError(\'Expected an existing Python import name\')\n    parts = module.split(\'.\')\n    found = []\n    for layout in (root, root / \'src\'):\n        package = layout.joinpath(*parts)\n        for candidate in (package / \'__init__.py\', package.with_suffix(\'.py\')):\n            if candidate.is_file():\n                resolved = candidate.resolve(strict=True)\n                if not resolved.is_relative_to(root):\n                    raise ValueError(\'Source symlink leaves workspace\')\n                found.append(resolved)\n    if len(found) != 1:\n        raise ValueError(\'Import must identify one source file in flat or src layout\')\n    return found[0]\n\n\ndef clean_env():\n    env = dict(os.environ)\n    env.update(PYTHONDONTWRITEBYTECODE=\'1\', PYTHONNOUSERSITE=\'1\',\n               PIP_NO_INDEX=\'1\', PIP_DISABLE_PIP_VERSION_CHECK=\'1\')\n    return env\n\n\ndef inventory(root, module, expected):\n    # A fresh cache prefix prevents old .pyc files from hiding rapid source edits.\n    code = \'\'\'import importlib,json,sys\ntry:\n m=importlib.import_module(sys.argv[1]); origin=getattr(m,"__file__",None)\n result={"import_succeeded":True,"origin":origin,"version":str(getattr(m,"__version__",""))}\nexcept Exception as e:\n result={"import_succeeded":False,"error_type":type(e).__name__,"error":str(e)[:500]}\nresult.update(executable=sys.executable,prefix=sys.prefix,base_prefix=sys.base_prefix)\nprint("SOURCE_IMPORT_INVENTORY="+json.dumps(result))\n\'\'\'\n    with tempfile.TemporaryDirectory(prefix=\'source_guard_cache_\') as cache:\n        proc = subprocess.run([sys.executable, \'-B\', \'-s\', \'-X\', \'pycache_prefix=\' + cache,\n                               \'-c\', code, module], cwd=root, env=clean_env(),\n                              capture_output=True, timeout=15, check=False)\n    lines = [line for line in proc.stdout.decode(errors=\'replace\').splitlines()\n             if line.startswith(\'SOURCE_IMPORT_INVENTORY=\')]\n    if proc.returncode != 0 or len(lines) != 1:\n        raise ValueError(\'Fresh interpreter import inventory failed\')\n    result = json.loads(lines[0].split(\'=\', 1)[1])\n    origin = result.get(\'origin\')\n    result[\'imports_expected_source\'] = bool(origin and Path(origin).resolve() == expected)\n    result[\'isolated_venv\'] = result[\'prefix\'] != result[\'base_prefix\']\n    return result\n\n\ndef backend_available(root):\n    backend = \'setuptools.build_meta\'\n    metadata = root / \'pyproject.toml\'\n    if metadata.is_file():\n        import tomllib\n        config = tomllib.loads(metadata.read_text(encoding=\'utf-8\'))\n        build = config.get(\'build-system\', {})\n        if build.get(\'backend-path\'):\n            raise ValueError(\'In-tree build backend requires manual review\')\n        backend = build.get(\'build-backend\', backend).split(\':\', 1)[0]\n    elif not (root / \'setup.py\').is_file():\n        raise ValueError(\'No declared editable project metadata\')\n    try:\n        available = importlib.util.find_spec(backend) is not None\n    except (ImportError, ValueError):\n        available = False\n    return backend, available\n\n\ndef generated_metadata(root):\n    return {p.resolve() for base in (root, root / \'src\') if base.is_dir()\n            for p in base.glob(\'*.egg-info\') if p.is_dir() and not p.is_symlink()}\n\n\ndef ensure_source(root, module, *, install=False, timeout=45):\n    started = time.monotonic()\n    expected = source_location(root, module)\n    before_diff = git(root, \'diff\', \'--binary\', \'HEAD\')\n    if before_diff.returncode:\n        raise ValueError(\'Cannot inspect source diff\')\n    report = {\'created_utc\': datetime.now(timezone.utc).isoformat(),\n              \'workspace\': str(root), \'module\': module, \'expected_source\': str(expected),\n              \'scope\': \'Agent-side import provenance, not repair correctness or grader equivalence\',\n              \'network_or_dependency_resolution_enabled\': False,\n              \'installation_attempted\': False, \'before\': inventory(root, module, expected)}\n    if report[\'before\'][\'imports_expected_source\']:\n        report.update(status=\'already_imports_source\', after=report[\'before\'])\n    elif not install:\n        report[\'status\'] = \'source_not_imported\'\n    elif not report[\'before\'][\'isolated_venv\']:\n        report[\'status\'] = \'installation_refused_nonisolated_interpreter\'\n    else:\n        backend, available = backend_available(root)\n        report.update(build_backend=backend, build_backend_available=available)\n        if not available:\n            report[\'status\'] = \'build_backend_unavailable_offline\'\n        else:\n            existing = generated_metadata(root)\n            command = [sys.executable, \'-m\', \'pip\', \'install\', \'--no-index\',\n                       \'--no-build-isolation\', \'--no-deps\', \'-e\', str(root)]\n            report.update(installation_attempted=True, installation_command=command)\n            try:\n                proc = subprocess.run(command, cwd=root, env=clean_env(), capture_output=True,\n                                      timeout=timeout, check=False)\n                report.update(pip_exit_code=proc.returncode,\n                              pip_stdout_sha256=hashlib.sha256(proc.stdout).hexdigest(),\n                              pip_stderr_sha256=hashlib.sha256(proc.stderr).hexdigest())\n                if proc.returncode:\n                    report[\'pip_error_tail\'] = proc.stderr.decode(errors=\'replace\')[-1200:]\n                report[\'status\'] = \'installation_failed\' if proc.returncode else \'installed\'\n            except subprocess.TimeoutExpired:\n                report[\'status\'] = \'installation_timeout\'\n            # Remove only new build-generated metadata absent from tracked source.\n            cleaned = []\n            for path in generated_metadata(root) - existing:\n                resolved = path.resolve(strict=True)\n                if not resolved.is_relative_to(root) or resolved.suffix != \'.egg-info\':\n                    raise ValueError(\'Generated metadata cleanup leaves workspace\')\n                tracked = git(root, \'ls-files\', \'--\', path.relative_to(root).as_posix())\n                if tracked.returncode != 0 or tracked.stdout:\n                    continue\n                shutil.rmtree(resolved)\n                cleaned.append(path.relative_to(root).as_posix())\n            report[\'new_generated_metadata_removed\'] = sorted(cleaned)\n            report[\'after\'] = inventory(root, module, expected)\n            if report[\'status\'] == \'installed\':\n                report[\'status\'] = (\'source_import_verified\' if report[\'after\'][\'imports_expected_source\']\n                                    else \'installation_did_not_select_source\')\n    after_diff = git(root, \'diff\', \'--binary\', \'HEAD\')\n    report[\'tracked_diff_unchanged\'] = after_diff.returncode == 0 and after_diff.stdout == before_diff.stdout\n    report[\'elapsed_seconds\'] = round(time.monotonic() - started, 4)\n    if not report[\'tracked_diff_unchanged\']:\n        report[\'status\'] = \'setup_changed_tracked_source\'\n    report[\'success\'] = (report[\'status\'] in (\'already_imports_source\', \'source_import_verified\')\n                         and report[\'tracked_diff_unchanged\'])\n    return report\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--workspace\', type=Path)\n    parser.add_argument(\'--module\', required=True)\n    parser.add_argument(\'--ensure\', action=\'store_true\')\n    parser.add_argument(\'--install-timeout\', type=int, default=45, choices=range(1, 61))\n    args = parser.parse_args(argv)\n    try:\n        root = discover_workspace(args.workspace)\n        report = ensure_source(root, args.module, install=args.ensure, timeout=args.install_timeout)\n    except Exception as error:\n        report = {\'status\': \'environment_check_failed\', \'success\': False,\n                  \'error_type\': type(error).__name__, \'error\': str(error)[:700]}\n    print(\'SOURCE_IMPORT_GUARD=\' + json.dumps(report, sort_keys=True))\n    return 0 if report[\'success\'] else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n', 'fresh_import_feedback_replay.py': '"""Fresh eight-condition import/feedback experiment, without competition data.\n\nUsage: python fresh_import_feedback_replay.py --guard source_import_guard.py\nRequires Python 3.11+ and Git; creates an isolated local interpreter without\ninstalling packages. The same assertion runs in every condition.\n"""\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport itertools\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport tempfile\n\nMODULE = \'fresh_feedback_control\'\nCHECK = \'\'\'import hashlib,importlib,json,pathlib\nm=importlib.import_module("fresh_feedback_control")\np=pathlib.Path(m.__file__).resolve()\nprint("FRESH_FEEDBACK="+json.dumps({"origin":str(p),"version":m.__version__,\n    "value":m.VALUE,"source_sha256":hashlib.sha256(p.read_bytes()).hexdigest()}))\nassert m.VALUE == "after!"\n\'\'\'\n\n\ndef digest(raw):\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef source(value):\n    return f\'__version__ = "0.0.1"\\nVALUE = "{value}"\\n\'.encode()\n\n\ndef run_checked(args, **kwargs):\n    return subprocess.run([str(a) for a in args], capture_output=True,\n                          timeout=45, check=True, **kwargs)\n\n\ndef run(guard, plan=None):\n    grid = list(itertools.product((\'before\', \'after!\'), (\'before\', \'after!\'),\n                                  (\'foreign\', \'workspace\')))\n    if plan:\n        frozen = json.loads(plan.read_bytes())\n        if (frozen[\'guard_sha256\'] != digest(guard.read_bytes())\n                or frozen[\'driver_sha256\'] != digest(Path(__file__).read_bytes())\n                or frozen[\'assertion_sha256\'] != digest(CHECK.encode())\n                or frozen[\'conditions\'] != [list(c) for c in grid]):\n            raise ValueError(\'Frozen experiment identity changed\')\n    env = dict(os.environ, PYTHONDONTWRITEBYTECODE=\'1\', PYTHONNOUSERSITE=\'1\',\n               PIP_NO_INDEX=\'1\', PIP_DISABLE_PIP_VERSION_CHECK=\'1\')\n    env.pop(\'PYTHONPATH\', None)\n    observations = []\n    with tempfile.TemporaryDirectory(prefix=\'fresh_import_feedback_\') as temporary:\n        root = Path(temporary)\n        virtual = root / \'environment\'\n        run_checked([sys.executable, \'-m\', \'venv\', \'--without-pip\', virtual], env=env)\n        python = virtual / (\'Scripts/python.exe\' if os.name == \'nt\' else \'bin/python\')\n        repo = root / \'checkout\'\n        package = repo / \'src\' / MODULE\n        package.mkdir(parents=True)\n        local_file = package / \'__init__.py\'\n        local_file.write_bytes(source(\'before\'))\n        run_checked([\'git\', \'init\', \'-q\', repo])\n        run_checked([\'git\', \'-C\', repo, \'add\', \'.\'])\n        run_checked([\'git\', \'-C\', repo, \'-c\', \'user.name=Feedback Control\',\n                     \'-c\', \'user.email=control@example.invalid\',\n                     \'commit\', \'-qm\', \'synthetic baseline\'])\n        foreign = root / \'foreign\'\n        foreign.mkdir()\n        foreign_file = foreign / (MODULE + \'.py\')\n        check_path = root / \'fixed_assertion.py\'\n        check_path.write_bytes(CHECK.encode())\n        for checkout_value, foreign_value, selected in grid:\n            local_file.write_bytes(source(checkout_value))\n            foreign_file.write_bytes(source(foreign_value))\n            selected_path = foreign if selected == \'foreign\' else repo / \'src\'\n            condition_env = dict(env, PYTHONPATH=str(selected_path))\n            before_diff = run_checked([\'git\', \'-C\', repo, \'diff\', \'--binary\', \'HEAD\']).stdout\n            inspected = subprocess.run([str(python), str(guard), \'--workspace\', str(repo),\n                \'--module\', MODULE], env=condition_env, capture_output=True, timeout=30)\n            lines = [l for l in inspected.stdout.decode().splitlines()\n                     if l.startswith(\'SOURCE_IMPORT_GUARD=\')]\n            if len(lines) != 1:\n                raise ValueError(\'Missing unique native guard receipt\')\n            report = json.loads(lines[0].split(\'=\', 1)[1])\n            with tempfile.TemporaryDirectory(prefix=\'feedback_cache_\') as cache:\n                checked = subprocess.run([str(python), \'-B\', \'-s\', \'-X\',\n                    \'pycache_prefix=\' + cache, str(check_path)], cwd=repo,\n                    env=condition_env, capture_output=True, timeout=15)\n            rows = [l for l in checked.stdout.decode().splitlines()\n                    if l.startswith(\'FRESH_FEEDBACK=\')]\n            if len(rows) != 1:\n                raise ValueError(\'Test process failed before its import inventory\')\n            actual = json.loads(rows[0].split(\'=\', 1)[1])\n            expected_file = foreign_file if selected == \'foreign\' else local_file\n            expected_value = foreign_value if selected == \'foreign\' else checkout_value\n            expected_exit = 0 if expected_value == \'after!\' else 1\n            checkout_passes = checkout_value == \'after!\'\n            after_diff = run_checked([\'git\', \'-C\', repo, \'diff\', \'--binary\', \'HEAD\']).stdout\n            if (Path(actual[\'origin\']) != expected_file.resolve()\n                    or actual[\'source_sha256\'] != digest(expected_file.read_bytes())\n                    or actual[\'version\'] != \'0.0.1\' or actual[\'value\'] != expected_value\n                    or checked.returncode != expected_exit\n                    or report[\'success\'] != (selected == \'workspace\')\n                    or inspected.returncode != (0 if selected == \'workspace\' else 2)\n                    or report[\'installation_attempted\']\n                    or not report[\'tracked_diff_unchanged\'] or before_diff != after_diff\n                    or check_path.read_bytes() != CHECK.encode()):\n                raise ValueError(\'Origin, assertion, or unchanged-source control failed\')\n            observations.append(dict(checkout_value=checkout_value,\n                foreign_value=foreign_value, imported_origin=selected,\n                identical_version=\'0.0.1\', test_exit_code=checked.returncode,\n                expected_checkout_test_pass=checkout_passes,\n                misleading_feedback=(checked.returncode == 0) != checkout_passes,\n                false_accept=checked.returncode == 0 and not checkout_passes,\n                false_reject=checked.returncode != 0 and checkout_passes,\n                guard_accepts_provenance=report[\'success\'],\n                guard_status=report[\'status\'], tracked_diff_unchanged_by_guard=True,\n                imported_source_sha256=actual[\'source_sha256\'],\n                fixed_assertion_sha256=digest(CHECK.encode()),\n                test_stderr_sha256=digest(checked.stderr)))\n    return dict(created_utc=datetime.now(timezone.utc).isoformat(), status=\'completed\',\n        scope=\'Eight factorial conditions on one synthetic repository; not eight repair tasks\',\n        python_version=platform.python_version(), platform=platform.platform(),\n        git_version=run_checked([\'git\', \'--version\']).stdout.decode().strip(),\n        guard_sha256=digest(guard.read_bytes()),\n        driver_sha256=digest(Path(__file__).read_bytes()),\n        plan_sha256=digest(plan.read_bytes()) if plan else None,\n        observations=observations, assigned_conditions=8, completed_conditions=len(observations),\n        false_accept_conditions=sum(r[\'false_accept\'] for r in observations),\n        false_reject_conditions=sum(r[\'false_reject\'] for r in observations),\n        foreign_origins_rejected=sum(not r[\'guard_accepts_provenance\'] for r in observations),\n        all_same_assertion_version_and_interpreter=True,\n        environment_installations=0, network_competition_data_model_or_verifier_used=False,\n        model_repair_improvement_established=False)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--guard\', type=Path, default=Path(__file__).with_name(\'source_import_guard.py\'))\n    parser.add_argument(\'--plan\', type=Path)\n    parser.add_argument(\'--output\', type=Path)\n    args = parser.parse_args()\n    result = run(args.guard.resolve(strict=True), args.plan.resolve(strict=True) if args.plan else None)\n    if args.output:\n        with args.output.open(\'x\', encoding=\'utf-8\') as stream:\n            stream.write(json.dumps(result, indent=2) + \'\\n\')\n    print(json.dumps(result, indent=2))\n', 'fresh_import_feedback_plan.json': '{\r\n  "declared_utc": "2026-10-06T19:34:41.616254+00:00",\r\n  "scope": "Fresh factorial mechanism experiment, not competition repair evaluation",\r\n  "driver_sha256": "759c78d1251133a1c5fd08d81428ed723a914c7bf48a5d1ee052fb513a01d17a",\r\n  "guard_sha256": "430fdc3dc241952d57f6b57240943fff8ff3847172b537aef6b6e28357b83618",\r\n  "assertion_sha256": "9302752d1c3d84e110b482b5a93a845bd5097570348a793e51c1fb6feb2e90d9",\r\n  "conditions": [\r\n    [\r\n      "before",\r\n      "before",\r\n      "foreign"\r\n    ],\r\n    [\r\n      "before",\r\n      "before",\r\n      "workspace"\r\n    ],\r\n    [\r\n      "before",\r\n      "after!",\r\n      "foreign"\r\n    ],\r\n    [\r\n      "before",\r\n      "after!",\r\n      "workspace"\r\n    ],\r\n    [\r\n      "after!",\r\n      "before",\r\n      "foreign"\r\n    ],\r\n    [\r\n      "after!",\r\n      "before",\r\n      "workspace"\r\n    ],\r\n    [\r\n      "after!",\r\n      "after!",\r\n      "foreign"\r\n    ],\r\n    [\r\n      "after!",\r\n      "after!",\r\n      "workspace"\r\n    ]\r\n  ],\r\n  "condition_axes": [\r\n    "checkout source value",\r\n    "foreign source value",\r\n    "selected import origin"\r\n  ],\r\n  "unchanged_across_conditions": [\r\n    "assertion bytes",\r\n    "interpreter",\r\n    "reported version0.0.1",\r\n    "guard bytes"\r\n  ],\r\n  "stopping_rule": "Any mismatch in fixed assertion, expected origin, module SHA, guard result or diff preservation stops the experiment",\r\n  "assigned_conditions": 8,\r\n  "synthetic_repositories": 1,\r\n  "dependency_installation_or_network_enabled": false,\r\n  "competition_data_model_or_verifier_used": false,\r\n  "actual_execution_observed": false\r\n}\r\n'}
HASHES={'source_import_guard.py': '430fdc3dc241952d57f6b57240943fff8ff3847172b537aef6b6e28357b83618', 'fresh_import_feedback_replay.py': '759c78d1251133a1c5fd08d81428ed723a914c7bf48a5d1ee052fb513a01d17a', 'fresh_import_feedback_plan.json': '147ab4e43689f7abc3287eae35102419fe8a19565ed84d8fcf35f16bffad5ea0'}
for name,text in PAYLOAD.items():
    data=text.encode()
    if hashlib.sha256(data).hexdigest()!=HASHES[name]:
        raise ValueError('Frozen source mismatch')
    (ROOT/name).write_bytes(data)
result=subprocess.run([sys.executable,str(ROOT/'fresh_import_feedback_replay.py'),
    '--guard',str(ROOT/'source_import_guard.py'),
    '--plan',str(ROOT/'fresh_import_feedback_plan.json'),
    '--output',str(ROOT/'actual_feedback.json')],capture_output=True,text=True,timeout=120)
(ROOT/'stdout.txt').write_text(result.stdout)
(ROOT/'stderr.txt').write_text(result.stderr)
if result.returncode:
    raise RuntimeError('Frozen fresh experiment failed; preserve output before diagnosis')
actual=json.loads((ROOT/'actual_feedback.json').read_bytes())
if actual['completed_conditions']!=8 or actual['false_accept_conditions']!=1 or actual['false_reject_conditions']!=1 or actual['foreign_origins_rejected']!=4:
    raise ValueError('Fresh factorial observations differ from fixed expectations')
receipt=dict(status='completed',scope='Unchanged synthetic factorial portability replay, not native grader or model inference',
    frozen_member_sha256=HASHES,system=platform.system(),python_version=platform.python_version(),
    completed_conditions=8,actual_feedback_sha256=hashlib.sha256((ROOT/'actual_feedback.json').read_bytes()).hexdigest(),
    network_competition_data_model_or_verifier_used=False)
(ROOT/'portability_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
print(json.dumps(receipt,indent=2))
